In [1]:
# IMPORTS

import os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import statsmodels.api as sm
from sklearn.inspection import permutation_importance
import shap
import warnings
warnings.filterwarnings('ignore')

In [2]:
# CONFIG

CONFIG = {
    "data_path": "/kaggle/input/competitions/optiver-trading-at-the-close/train.csv",
    "output_dir": "outputs",
    "train_frac": 0.70,
    "val_frac": 0.15,
    "epsilon": 1e-9,
    "n_walk_forward_folds": 4,
    "n_bins_rq1": 7,
    "rq2_window_size": 60,
    "ridge_alpha": 1.0,
    "lgbm_params": {"objective": "regression_l1", "n_estimators": 500, "learning_rate": 0.05, "num_leaves": 63, "min_child_samples": 100, "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8, "n_jobs": -1, "verbosity": -1,},
    "lgbm_early_stopping_rounds": 50,
    "interaction_signal_col": "liquidity_imbalance",
    "temporal_lag_steps": 1,      
    "temporal_vol_window": 6,           
    "shap_sample_size": 20000,
    "shap_random_state": 42,
    "n_permutation_repeats": 10,
    "permutation_sample_size": 200000,  
    "shap_focus_signals": ["auction_imbalance_ratio", "liquidity_imbalance", "matched_size_ratio", "market_urgency"],
    "shap_top_n_features": 10,     
    "regime_n_tiers": 3,
    "regime_significance_level": 0.05,}

os.makedirs(CONFIG["output_dir"], exist_ok=True)

In [3]:
# DATA FORENSICS

def dataset_forensics(df, output_dir):
    print(f"Shape: {df.shape}")
    n_stocks = df["stock_id"].nunique()
    n_dates = df["date_id"].nunique()
    n_obs = len(df)
    print(f"\nN stocks      : {n_stocks}")
    print(f"N dates       : {n_dates}")
    print(f"N observations: {n_obs}")

    # structural uniqueness of (stock_id, date_id, seconds_in_bucket)
    key_cols = ["stock_id", "date_id", "seconds_in_bucket"]
    n_unique_keys = df.drop_duplicates(subset=key_cols).shape[0]
    print("Unique keys:", n_unique_keys)
    print("Rows:", n_obs)

    # missingness
    missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
    missing_pct.to_csv(os.path.join(output_dir, "missingness_by_column.csv"))

    # far_price / near_price are expected to be NaN early in the auction
    near_missing_by_bucket = df.groupby("seconds_in_bucket")["near_price"].apply(lambda s: s.isna().mean())
    print("\nnear_price missing rate by seconds_in_bucket (first/last 5 buckets):")
    print(near_missing_by_bucket.head().to_string())
    print(near_missing_by_bucket.tail().to_string())

    # coverage: observations per stock / per date
    obs_per_stock = df.groupby("stock_id").size()
    obs_per_date = df.groupby("date_id").size()
    obs_per_stock.to_csv(os.path.join(output_dir, "observations_per_stock.csv"))
    obs_per_date.to_csv(os.path.join(output_dir, "observations_per_date.csv"))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(obs_per_stock, bins=30)
    axes[0].set_title("Observations per stock")
    axes[1].hist(obs_per_date, bins=30)
    axes[1].set_title("Observations per date")
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, "coverage_histograms.png"), dpi=120)
    plt.close(fig)

    # target distribution
    if "target" in df.columns:
        tgt = df["target"].dropna()
        desc = tgt.describe()
        skew = stats.skew(tgt)
        kurt = stats.kurtosis(tgt)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.hist(tgt.clip(tgt.quantile(0.001), tgt.quantile(0.999)), bins=100)
        ax.set_title("Target distribution (0.1%-99.9%)")
        ax.set_xlabel("target (bps)")
        fig.tight_layout()
        fig.savefig(os.path.join(output_dir, "target_distribution.png"), dpi=120)
        plt.close(fig)
        # target by seconds_in_bucket (predictability vs closeness)
        by_bucket = df.groupby("seconds_in_bucket")["target"].agg(["mean", "std", "count"])
        by_bucket.to_csv(os.path.join(output_dir, "target_by_seconds_in_bucket.csv"))
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(by_bucket.index, by_bucket["mean"])
        axes[0].axhline(0, color="grey", lw=0.8)
        axes[0].set_title("Mean target vs seconds_in_bucket")
        axes[0].set_xlabel("seconds_in_bucket")
        axes[1].plot(by_bucket.index, by_bucket["std"])
        axes[1].set_title("Std(target) vs seconds_in_bucket\n(volatility as close approaches)")
        axes[1].set_xlabel("seconds_in_bucket")
        fig.tight_layout()
        fig.savefig(os.path.join(output_dir, "target_vs_seconds_in_bucket.png"), dpi=120)
        plt.close(fig)
        # target by stock 
        by_stock = df.groupby("stock_id")["target"].agg(["mean", "std", "count"]).sort_values("mean")
        by_stock.to_csv(os.path.join(output_dir, "target_by_stock.csv"))
        # extreme observations
        q1, q3 = tgt.quantile(0.25), tgt.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 3 * iqr, q3 + 3 * iqr
        extreme_mask = (df["target"] < lo) | (df["target"] > hi)
        n_extreme = int(extreme_mask.sum())
        print(f"\nExtreme observations: {n_extreme} " f"({100 * n_extreme / len(df):.3f}% of rows), bounds=({lo:.2f}, {hi:.2f})\n")
        # Feature distributions
        cols = [c for c in df.columns if c not in ["stock_id", "date_id", "seconds_in_bucket"]]

In [4]:
# VALIDATION FRAMEWORK

def chronological_split(df, train_frac=0.70, val_frac=0.15, date_col="date_id"):
    unique_dates = np.sort(df[date_col].unique())
    n = len(unique_dates)
    n_train = int(np.floor(n * train_frac))
    n_val = int(np.floor(n * val_frac))
    train_dates = unique_dates[:n_train]
    val_dates = unique_dates[n_train:n_train + n_val]
    test_dates = unique_dates[n_train + n_val:]
    train_df = df[df[date_col].isin(train_dates)].copy()
    val_df = df[df[date_col].isin(val_dates)].copy()
    test_df = df[df[date_col].isin(test_dates)].copy()
    print(f"Chronological split:")
    print(f"train dates: [{train_dates.min()}, {train_dates.max()}]")
    print(f"val dates: [{val_dates.min()}, {val_dates.max()}]")
    print(f"test dates: [{test_dates.min()}, {test_dates.max()}]\n")
    return train_df, val_df, test_df

def generate_walk_forward_folds(df, date_col="date_id", n_folds=4, initial_train_frac=0.5, val_span_frac=0.10):
    unique_dates = np.sort(df[date_col].unique())
    n = len(unique_dates)
    initial_train_end = int(np.floor(n * initial_train_frac))
    val_span = max(1, int(np.floor(n * val_span_frac)))
    fold_id = 0
    train_end = initial_train_end
    while train_end + val_span <= n and fold_id < n_folds:
        fold_id += 1
        train_dates = unique_dates[:train_end]
        val_dates = unique_dates[train_end:train_end + val_span]
        train_df = df[df[date_col].isin(train_dates)]
        val_df = df[df[date_col].isin(val_dates)]
        print(f"Fold {fold_id}: train dates [{train_dates.min()},{train_dates.max()}] " f"({len(train_dates)} dates) -> val dates [{val_dates.min()},{val_dates.max()}] " f"({len(val_dates)} dates)")
        yield fold_id, train_df, val_df
        train_end += val_span

In [5]:
# BASELINES

def evaluate_predictions(y_true, y_pred, name=""):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true, y_pred = y_true[mask], y_pred[mask]
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    if np.std(y_pred) > 0 and np.std(y_true) > 0:
        corr = np.corrcoef(y_true, y_pred)[0, 1]
    else:
        corr = np.nan
    true_sign = np.sign(y_true)
    pred_sign = np.sign(y_pred)
    nonzero_mask = pred_sign != 0
    if nonzero_mask.sum() > 0:
        dir_acc = np.mean(true_sign[nonzero_mask] == pred_sign[nonzero_mask])
    else:
        dir_acc = np.nan
    return {"model": name, "MAE": mae, "RMSE": rmse, "corr": corr, "directional_accuracy": dir_acc, "n": int(mask.sum()),}

def baseline_zero(train_df, eval_df, target_col="target"):
    return np.zeros(len(eval_df))

def baseline_historical_mean(train_df, eval_df, target_col="target"):
    mu = train_df[target_col].mean()
    return np.full(len(eval_df), mu)

def baseline_stock_mean(train_df, eval_df, target_col="target", stock_col="stock_id"):
    stock_means = train_df.groupby(stock_col)[target_col].mean()
    overall_mean = train_df[target_col].mean()
    return eval_df[stock_col].map(stock_means).fillna(overall_mean).values

def baseline_time_bucket_mean(train_df, eval_df, target_col="target", time_col="seconds_in_bucket"):
    bucket_means = train_df.groupby(time_col)[target_col].mean()
    overall_mean = train_df[target_col].mean()
    return eval_df[time_col].map(bucket_means).fillna(overall_mean).values

def run_all_baselines(train_df, eval_df, target_col="target"):
    baselines = {"zero": baseline_zero, "historical_mean": baseline_historical_mean, "stock_mean": baseline_stock_mean, "time_bucket_mean": baseline_time_bucket_mean,}
    rows = []
    y_true = eval_df[target_col].values
    for name, fn in baselines.items():
        y_pred = fn(train_df, eval_df, target_col=target_col)
        rows.append(evaluate_predictions(y_true, y_pred, name=name))
    result = pd.DataFrame(rows).set_index("model")
    return result

In [6]:
# INTERPRETABLE MICROSTRUCTURE FEATURES

def add_microstructure_features(df, eps=CONFIG["epsilon"]):
    df = df.copy()
    df["mid_price"] = (df["ask_price"] + df["bid_price"]) / 2
    df["spread"] = df["ask_price"] - df["bid_price"]
    # order-book (non-auction) liquidity imbalance in [-1, 1]
    df["liquidity_imbalance"] = ((df["bid_size"] - df["ask_size"]) / (df["bid_size"] + df["ask_size"] + eps))
    df["market_urgency"] = df["spread"] * df["liquidity_imbalance"]
    # auction imbalance, signed by the direction flag, scaled by matched size
    df["signed_imbalance_size"] = df["imbalance_size"] * df["imbalance_buy_sell_flag"]
    df["auction_imbalance_ratio"] = df["signed_imbalance_size"] / (df["matched_size"] + eps)
    # how much of total auction interest is actually matchable right now
    df["matched_size_ratio"] = df["matched_size"] / (df["matched_size"] + df["imbalance_size"] + eps)
    # normalized price distances
    df["ref_mid_dist"] = (df["reference_price"] - df["mid_price"]) / (df["mid_price"] + eps)
    df["far_near_dist"] = (df["far_price"] - df["near_price"]) / (df["mid_price"] + eps)
    return df

def feature_engineering_report(df, output_dir):
    new_cols = ["mid_price", "spread", "liquidity_imbalance", "market_urgency", "signed_imbalance_size", "auction_imbalance_ratio", "matched_size_ratio", "ref_mid_dist", "far_near_dist"]
    desc = df[new_cols].describe().T
    desc.to_csv(os.path.join(output_dir, "engineered_feature_describe.csv"))
    return desc

In [7]:
# H1

BIN_LABELS = ["very negative", "negative", "slightly negative", "neutral", "slightly positive", "positive", "very positive",]

def bin_feature_quantile(series, n_bins=7):
    labels = BIN_LABELS[:n_bins]
    s = series.copy()
    try:
        binned = pd.qcut(s, q=n_bins, labels=labels, duplicates="drop")
    except ValueError:
        for k in range(n_bins - 1, 1, -1):
            try:
                binned = pd.qcut(s, q=k, labels=BIN_LABELS[:k] if k == n_bins else None, duplicates="drop")
                break
            except ValueError:
                continue
    return binned

def analyze_signal_vs_target(df, feature_col, target_col="target", n_bins=7):
    sub = df[[feature_col, target_col]].dropna()
    n = len(sub)
    bins = bin_feature_quantile(sub[feature_col], n_bins=n_bins)
    grouped = sub.groupby(bins)[target_col].agg(["mean", "std", "count"])
    grouped["sem"] = grouped["std"] / np.sqrt(grouped["count"])
    grouped["ci95_low"] = grouped["mean"] - 1.96 * grouped["sem"]
    grouped["ci95_high"] = grouped["mean"] + 1.96 * grouped["sem"]
    bin_edges = sub.groupby(bins)[feature_col].agg(["min", "max"])
    grouped = grouped.join(bin_edges.rename(columns={"min": "feature_min", "max": "feature_max"}))
    spearman_r, spearman_p = stats.spearmanr(sub[feature_col], sub[target_col])
    pearson_r, pearson_p = stats.pearsonr(sub[feature_col], sub[target_col])
    groups = [g[target_col].values for _, g in sub.groupby(bins) if len(g) > 1]
    if len(groups) > 1:
        f_stat, anova_p = stats.f_oneway(*groups)
    else:
        f_stat, anova_p = np.nan, np.nan
    stats_dict = {"feature": feature_col, "n": n, "spearman_r": spearman_r, "spearman_p": spearman_p, "pearson_r": pearson_r, "pearson_p": pearson_p, "anova_f": f_stat, "anova_p": anova_p,}
    return grouped, stats_dict

def plot_signal_vs_target(grouped, feature_name, output_path):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(grouped))
    means = grouped["mean"].values
    err = (grouped["ci95_high"] - grouped["mean"]).values
    ax.bar(x, means, yerr=err, capsize=4, color="#3b6fa0")
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(grouped.index.astype(str), rotation=30, ha="right")
    ax.set_ylabel("E[target | bin]  (bps, with 95% CI)")
    ax.set_title(f"Future return vs {feature_name}")
    fig.tight_layout()
    fig.savefig(output_path, dpi=120)
    plt.close(fig)

def run_rq1_experiment(df, output_dir, n_bins=7):
    signals = {
        "liquidity_imbalance": "order-book liquidity imbalance (bid vs ask size)",
        "auction_imbalance_ratio": "auction imbalance, signed and scaled by matched size",
        "matched_size_ratio": "share of auction interest that is currently matchable",
        "market_urgency": "spread x liquidity imbalance",}

    summary_rows = []
    for col, description in signals.items():
        grouped, stats_dict = analyze_signal_vs_target(df, col, n_bins=n_bins)
        grouped.to_csv(os.path.join(output_dir, f"rq1_{col}_binned_target.csv"))
        plot_signal_vs_target(grouped, col, os.path.join(output_dir, f"rq1_{col}_vs_target.png"))
        summary_rows.append(stats_dict)
    summary_df = pd.DataFrame(summary_rows).set_index("feature")
    summary_df.to_csv(os.path.join(output_dir, "rq1_significance_summary.csv"))
    return summary_df

In [8]:
# H2

def assign_time_window(df, window_size=60, time_col="seconds_in_bucket", max_time=540):
    df = df.copy()
    n_windows = int(np.ceil(max_time / window_size))
    window_idx = np.minimum((df[time_col] // window_size).astype(int), n_windows - 1)
    window_start = window_idx * window_size
    window_end = np.minimum(window_start + window_size, max_time)
    df["time_window_idx"] = window_idx
    df["time_window"] = [f"{s}-{e}" for s, e in zip(window_start, window_end)]
    return df

def analyze_time_dependence(train_df, val_df, signal_cols, target_col="target", window_size=60, max_time=540):
    train_w = assign_time_window(train_df, window_size, max_time=max_time)
    val_w = assign_time_window(val_df, window_size, max_time=max_time)
    window_order = (val_w[["time_window_idx", "time_window"]].drop_duplicates().sort_values("time_window_idx")["time_window"].tolist())
    rows = []
    for window_label in window_order:
        train_win = train_w[train_w["time_window"] == window_label]
        val_win = val_w[val_w["time_window"] == window_label]
        for sig in signal_cols:
            tr = train_win[[sig, target_col]].dropna()
            va = val_win[[sig, target_col]].dropna()
            pearson_r, pearson_p = stats.pearsonr(va[sig], va[target_col])
            spearman_r, spearman_p = stats.spearmanr(va[sig], va[target_col])
 
            # simple univariate OLS fit on train, scored out-of-sample on val
            slope, intercept = np.polyfit(tr[sig], tr[target_col], deg=1)
            pred = intercept + slope * va[sig]
            mae_model = float(np.mean(np.abs(va[target_col] - pred)))
 
            train_mean = tr[target_col].mean()
            mae_baseline = float(np.mean(np.abs(va[target_col] - train_mean)))
            mae_reduction_pct = (100.0 * (mae_baseline - mae_model) / mae_baseline if mae_baseline else np.nan)
 
            # conditional mean return: top-decile minus bottom-decile mean(target), on val
            deciles = pd.qcut(va[sig], 10, labels=False, duplicates="drop")
            top_mean = va.loc[deciles == deciles.max(), target_col].mean()
            bottom_mean = va.loc[deciles == deciles.min(), target_col].mean()
            decile_spread = top_mean - bottom_mean
            r2 = pearson_r ** 2
            snr = r2 / (1 - r2) if r2 < 1 else np.inf
            rows.append({
                "time_window": window_label,
                "window_start": int(window_label.split("-")[0]),
                "signal": sig,
                "n_train": len(tr),
                "n_val": len(va),
                "pearson_r": pearson_r,
                "pearson_p": pearson_p,
                "spearman_r": spearman_r,
                "spearman_p": spearman_p,
                "mae_model": mae_model,
                "mae_baseline": mae_baseline,
                "mae_reduction_pct": mae_reduction_pct,
                "decile_spread_bps": decile_spread,
                "snr": snr,})
    return pd.DataFrame(rows)

def plot_time_dependence(results_df, output_dir):
    signals = results_df["signal"].unique()
    metrics = [("pearson_r", "Pearson correlation (val)"), ("mae_reduction_pct", "MAE reduction vs. mean baseline (%)"), ("decile_spread_bps", "Top-decile minus bottom-decile\nmean target (bps)"), ("snr", "Signal-to-noise ratio  r^2/(1-r^2)"),]
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    axes = axes.flatten()
    for ax, (metric, ylabel) in zip(axes, metrics):
        for sig in signals:
            sub = results_df[results_df["signal"] == sig].sort_values("window_start")
            ax.plot(sub["window_start"], sub[metric], marker="o", label=sig)
        ax.axhline(0, color="grey", lw=0.8)
        ax.set_xlabel("seconds_in_bucket (window start)")
        ax.set_ylabel(ylabel)
        ax.set_title(ylabel)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=len(signals), bbox_to_anchor=(0.5, -0.02))
    fig.suptitle("RQ2 - Information content of imbalance signals vs. time-to-close", y=1.0)
    fig.tight_layout(rect=[0, 0.04, 1, 0.97])
    fig.savefig(os.path.join(output_dir, "rq2_time_dependence.png"), dpi=130, bbox_inches="tight")
    plt.close(fig)

def run_rq2_experiment(train_df, val_df, output_dir, window_size=60):
    signal_cols = ["liquidity_imbalance", "auction_imbalance_ratio", "matched_size_ratio", "market_urgency"]
    results_df = analyze_time_dependence(train_df, val_df, signal_cols, window_size=window_size)
    results_df.to_csv(os.path.join(output_dir, "rq2_time_dependence.csv"), index=False)
    plot_time_dependence(results_df, output_dir)
 
    # quick early-vs-late summary per signal (splits the 9 windows at the median)
    mid = results_df["window_start"].median()
    summary_rows = []
    for sig in signal_cols:
        sub = results_df[results_df["signal"] == sig]
        early = sub[sub["window_start"] < mid]
        late = sub[sub["window_start"] >= mid]
        summary_rows.append({
            "signal": sig,
            "early_abs_r_mean": early["pearson_r"].abs().mean(),
            "late_abs_r_mean": late["pearson_r"].abs().mean(),
            "early_snr_mean": early["snr"].mean(),
            "late_snr_mean": late["snr"].mean(),})
    early_late_df = pd.DataFrame(summary_rows).set_index("signal")
    early_late_df.to_csv(os.path.join(output_dir, "rq2_early_vs_late_summary.csv"))
    return results_df

In [9]:
# FEATURE SET DEFINITIONS

def add_time_features(df, max_time=540, late_threshold=300):
    df = df.copy()
    df["is_late_auction"] = (df["seconds_in_bucket"] >= late_threshold).astype(int)
    return df

def add_cross_sectional_features(df, cols=("liquidity_imbalance", "auction_imbalance_ratio", "market_urgency"), eps=CONFIG["epsilon"]):
    df = df.copy()
    grp = df.groupby(["date_id", "seconds_in_bucket"])
    for col in cols:
        mu = grp[col].transform("mean")
        sigma = grp[col].transform("std")
        df[f"{col}_cs_z"] = (df[col] - mu) / (sigma + eps)
        df[f"{col}_cs_rank"] = grp[col].rank(pct=True)
    return df
    
RAW_FEATURES = ["reference_price", "matched_size", "far_price", "near_price", "bid_price", "bid_size", "ask_price", "ask_size", "wap", "imbalance_size", "imbalance_buy_sell_flag",]
IMBALANCE_FEATURES = ["liquidity_imbalance", "auction_imbalance_ratio", "matched_size_ratio", "market_urgency", "ref_mid_dist", "far_near_dist",]
TIME_FEATURES = ["seconds_in_bucket", "is_late_auction"]
CROSS_SECTIONAL_FEATURES = ([f"{c}_cs_z" for c in ("liquidity_imbalance", "auction_imbalance_ratio", "market_urgency")] + [f"{c}_cs_rank" for c in ("liquidity_imbalance", "auction_imbalance_ratio", "market_urgency")])

def compute_group_stats(train_df, cols, group_col):
    return train_df.groupby(group_col)[list(cols)].agg(["mean", "std"])

def add_relative_zscore(df, cols, group_col, stats, eps=CONFIG["epsilon"]):
    df = df.copy()
    for col in cols:
        mu = df[group_col].map(stats[(col, "mean")])
        sigma = df[group_col].map(stats[(col, "std")])
        df[f"{col}_{group_col}_z"] = (df[col] - mu) / (sigma + eps)
    return df

# MODEL 1: RIDGE REGRESSION ACROSS FEATURE SETS

def prepare_features(train_df, val_df, feature_cols, target_col="target", standardize=True):
    train_df = train_df.dropna(subset=[target_col])
    val_df = val_df.dropna(subset=[target_col])
    train_X = train_df[feature_cols].copy()
    val_X = val_df[feature_cols].copy()
    medians = train_X.median()
    train_X = train_X.fillna(medians)
    val_X = val_X.fillna(medians)
    scaler = None
    if standardize:
        scaler = StandardScaler()
        train_arr = scaler.fit_transform(train_X.values)
        val_arr = scaler.transform(val_X.values)
    else:
        train_arr = train_X.values
        val_arr = val_X.values
    y_train = train_df[target_col].values
    y_val = val_df[target_col].values
    return train_arr, val_arr, y_train, y_val, medians, scaler

def fit_ridge_and_evaluate(train_df, val_df, feature_cols, feature_set_name, target_col="target", standardize=True, alpha=1.0): 
    X_train, X_val, y_train, y_val, medians, scaler = prepare_features(train_df, val_df, feature_cols, target_col=target_col, standardize=standardize)
    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)
    pred_val = model.predict(X_val)
    metrics = evaluate_predictions(y_val, pred_val, name=feature_set_name)
    coef_df = pd.DataFrame({"feature": feature_cols, "coefficient": model.coef_})
    coef_df = coef_df.reindex(coef_df["coefficient"].abs().sort_values(ascending=False).index)
    return metrics, coef_df, model, scaler, medians

def run_model1(train_df, val_df, output_dir):
    train_t = add_time_features(train_df)
    val_t = add_time_features(val_df)
    train_t = add_cross_sectional_features(train_t)
    val_t = add_cross_sectional_features(val_t)
    normalize_cols = ["spread", "matched_size", "imbalance_size"]
    stock_stats = compute_group_stats(train_t, normalize_cols, "stock_id")
    bucket_stats = compute_group_stats(train_t, normalize_cols, "seconds_in_bucket")
    train_t = add_relative_zscore(train_t, normalize_cols, "stock_id", stock_stats)
    val_t = add_relative_zscore(val_t, normalize_cols, "stock_id", stock_stats)
    train_t = add_relative_zscore(train_t, normalize_cols, "seconds_in_bucket", bucket_stats)
    val_t = add_relative_zscore(val_t, normalize_cols, "seconds_in_bucket", bucket_stats)
    stock_relative_cols = [f"{c}_stock_id_z" for c in normalize_cols]
    bucket_relative_cols = [f"{c}_seconds_in_bucket_z" for c in normalize_cols]
    feature_sets = {
        "raw": (RAW_FEATURES, False),
        "normalized": (RAW_FEATURES, True),
        "imbalance": (IMBALANCE_FEATURES, True),
        "time": (TIME_FEATURES, True),
        "cross_sectional": (CROSS_SECTIONAL_FEATURES, True),
        "stock_relative": (stock_relative_cols, True),
        "bucket_relative": (bucket_relative_cols, True),}

    all_metrics = []
    all_coefs = {}
    for name, (cols, standardize) in feature_sets.items():
        metrics, coef_df, model, scaler, medians = fit_ridge_and_evaluate(train_t, val_t, cols, name, standardize=standardize, alpha=CONFIG["ridge_alpha"],)
        all_metrics.append(metrics)
        all_coefs[name] = coef_df
        coef_df.to_csv(os.path.join(output_dir, f"model1_ridge_{name}_coefficients.csv"), index=False)

    metrics_df = pd.DataFrame(all_metrics).set_index("model")
    metrics_df.to_csv(os.path.join(output_dir, "model1_ridge_feature_set_comparison.csv"))
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(metrics_df.index, metrics_df["MAE"])
    ax.set_ylabel("MAE (bps)")
    ax.set_title("Ridge validation MAE by feature set")
    plt.xticks(rotation=20, ha="right")
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, "model1_mae_by_feature_set.png"), dpi=120)
    plt.close(fig)
    return metrics_df, all_coefs

In [10]:
# CUMULATIVE FEATURE-GROUP ABLATION (LIGHTGBM)

def run_cumulative_feature_ablation(train_df, val_df, output_dir):
    train_t = add_time_features(train_df)
    val_t = add_time_features(val_df)
    train_t = add_cross_sectional_features(train_t)
    val_t = add_cross_sectional_features(val_t)
    feature_groups = [("raw", RAW_FEATURES), ("+ imbalance", IMBALANCE_FEATURES), ("+ time", TIME_FEATURES), ("+ cross_sectional", CROSS_SECTIONAL_FEATURES),]
    cumulative_cols = []
    rows = []
    prev_mae = None
    print("\nCUMULATIVE FEATURE-GROUP ABLATION\n")
    for name, cols in feature_groups:
        cumulative_cols = cumulative_cols + cols
        train_valid = train_t[cumulative_cols + ["target"]].dropna(subset=["target"])
        val_valid = val_t[cumulative_cols + ["target"]].dropna(subset=["target"])
        X_train = train_valid[cumulative_cols]
        y_train = train_valid["target"].values
        X_val = val_valid[cumulative_cols]
        y_val = val_valid["target"].values
        model = lgb.LGBMRegressor(**CONFIG["lgbm_params"])
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="l1", callbacks=[lgb.early_stopping(CONFIG["lgbm_early_stopping_rounds"], verbose=False)],)
        pred_val = model.predict(X_val, num_iteration=model.best_iteration_)
        metrics = evaluate_predictions(y_val, pred_val, name=name)
        mae = metrics["MAE"]
        delta = mae - prev_mae if prev_mae is not None else np.nan
        rows.append({"feature_set": name, "n_features": len(cumulative_cols), "MAE": mae, "delta_MAE": delta})
        prev_mae = mae
        delta_str = f"  delta_MAE={delta:+.5f}" if not np.isnan(delta) else ""
        print(f"{name:<18} n_features={len(cumulative_cols):>3}  MAE={mae:.5f}{delta_str}")

    ablation_df = pd.DataFrame(rows).set_index("feature_set")
    ablation_df.to_csv(os.path.join(output_dir, "lgbm_feature_group_ablation.csv"))
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(ablation_df.index, ablation_df["MAE"], marker="o")
    ax.set_ylabel("Validation MAE (bps)")
    ax.set_title("Cumulative feature-group ablation (LightGBM)")
    plt.xticks(rotation=15, ha="right")
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, "lgbm_feature_group_ablation.png"), dpi=120)
    plt.close(fig)

    # the final model (every group included)
    importance_df = pd.DataFrame({"feature": cumulative_cols, "gain_importance": model.booster_.feature_importance(importance_type="gain"),}).sort_values("gain_importance", ascending=False)
    importance_df.to_csv(os.path.join(output_dir, "lgbm_full_feature_importance.csv"), index=False)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh(importance_df["feature"][::-1], importance_df["gain_importance"][::-1])
    ax.set_xlabel("Gain importance")
    ax.set_title("LightGBM feature importance (full feature set)")
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, "lgbm_full_feature_importance.png"), dpi=120)
    plt.close(fig)
    return ablation_df, importance_df, model

def run_interaction_ablation(train_df, val_df, signal_col, output_dir, time_col="seconds_in_bucket", target_col="target"):
    tr = train_df[[signal_col, time_col, target_col]].dropna()
    va = val_df[[signal_col, time_col, target_col]].dropna() 
    scaler = StandardScaler()
    tr_scaled = scaler.fit_transform(tr[[signal_col, time_col]].values)
    va_scaled = scaler.transform(va[[signal_col, time_col]].values)
    tr_sig, tr_time = tr_scaled[:, 0], tr_scaled[:, 1]
    va_sig, va_time = va_scaled[:, 0], va_scaled[:, 1]

    # A: additive only
    X_tr_a = np.column_stack([tr_sig, tr_time])
    X_va_a = np.column_stack([va_sig, va_time])
    model_a = Ridge(alpha=CONFIG["ridge_alpha"]).fit(X_tr_a, tr[target_col].values)
    metrics_a = evaluate_predictions(va[target_col].values, model_a.predict(X_va_a), name="A_additive_only")

    # B: additive + explicit interaction term
    X_tr_b = np.column_stack([tr_sig, tr_time, tr_sig * tr_time])
    X_va_b = np.column_stack([va_sig, va_time, va_sig * va_time])
    model_b = Ridge(alpha=CONFIG["ridge_alpha"]).fit(X_tr_b, tr[target_col].values)
    metrics_b = evaluate_predictions(va[target_col].values, model_b.predict(X_va_b), name="B_explicit_interaction")
    X_sm = sm.add_constant(X_tr_b)
    ols_model = sm.OLS(tr[target_col].values, X_sm).fit()
    interaction_coef = ols_model.params[-1]
    interaction_p = ols_model.pvalues[-1]

    # C: LightGBM on the same two RAW features 
    lgbm_params = dict(CONFIG["lgbm_params"])
    model_c = lgb.LGBMRegressor(**lgbm_params)
    model_c.fit(tr[[signal_col, time_col]], tr[target_col].values, eval_set=[(va[[signal_col, time_col]], va[target_col].values)], eval_metric="l1", callbacks=[lgb.early_stopping(CONFIG["lgbm_early_stopping_rounds"], verbose=False)],)
    pred_c = model_c.predict(va[[signal_col, time_col]], num_iteration=model_c.best_iteration_)
    metrics_c = evaluate_predictions(va[target_col].values, pred_c, name="C_lightgbm_2feature")

    comparison = pd.DataFrame([metrics_a, metrics_b, metrics_c]).set_index("model")
    mae_a = comparison.loc["A_additive_only", "MAE"]
    mae_b = comparison.loc["B_explicit_interaction", "MAE"]
    mae_c = comparison.loc["C_lightgbm_2feature", "MAE"]
    b_gain_pct = 100 * (mae_a - mae_b) / mae_a
    c_gain_pct = 100 * (mae_a - mae_c) / mae_a
    comparison.to_csv(os.path.join(output_dir, f"interaction_ablation_{signal_col}.csv"))

    # 2D heatmap: E[target | signal_bin, time_bin] on validation data
    va_binned = va.copy()
    va_binned["signal_bin"] = pd.qcut(va_binned[signal_col], 5, duplicates="drop")
    va_binned["time_bin"] = pd.qcut(va_binned[time_col], 9, duplicates="drop")
    heat = va_binned.groupby(["signal_bin", "time_bin"], observed=True)[target_col].mean().unstack()
    heat.to_csv(os.path.join(output_dir, f"interaction_heatmap_{signal_col}.csv"))
    vmax = np.nanmax(np.abs(heat.values))
    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(heat.values, aspect="auto", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(len(heat.columns)))
    ax.set_xticklabels([str(c) for c in heat.columns], rotation=30, ha="right")
    ax.set_yticks(range(len(heat.index)))
    ax.set_yticklabels([str(i) for i in heat.index])
    ax.set_xlabel(time_col)
    ax.set_ylabel(signal_col)
    ax.set_title(f"E[target | {signal_col} bin, {time_col} bin]")
    fig.colorbar(im, ax=ax, label="mean target (bps)")
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, f"interaction_heatmap_{signal_col}.png"), dpi=120)
    plt.close(fig)
    
    return {"comparison": comparison, "interaction_coef": interaction_coef, "interaction_p": interaction_p, "heatmap": heat,}


In [11]:
# TEMPORAL DYNAMICS

TEMPORAL_DYNAMICS_FEATURES = ["imbalance_delta", "wap_delta", "wap_return", "realized_vol", "imbalance_accel"]

def add_temporal_dynamics_features(df, lag_steps=CONFIG["temporal_lag_steps"], vol_window=CONFIG["temporal_vol_window"],group_cols=("stock_id", "date_id"), time_col="seconds_in_bucket"):
    group_cols = list(group_cols)
    out = df.copy()
    ordered = out.sort_values(group_cols + [time_col], kind="mergesort")
    def by_group(frame):
        return frame.groupby(group_cols, sort=False)
    ordered["imbalance_delta"] = by_group(ordered)["auction_imbalance_ratio"].diff(lag_steps)
    ordered["wap_delta"] = by_group(ordered)["wap"].diff(lag_steps)
    lagged_wap = by_group(ordered)["wap"].shift(lag_steps)
    ordered["wap_return"] = (ordered["wap"] / lagged_wap - 1.0).replace([np.inf, -np.inf], np.nan)
    ordered["realized_vol"] = (by_group(ordered)["wap_return"].rolling(vol_window, min_periods=2).std().droplevel(list(range(len(group_cols)))))
    ordered["imbalance_accel"] = by_group(ordered)["imbalance_delta"].diff(lag_steps)
    for col in TEMPORAL_DYNAMICS_FEATURES:
        out[col] = ordered[col]
    return out

def run_temporal_dynamics_experiment(train_df, val_df, output_dir, level_col="auction_imbalance_ratio", n_bins=CONFIG["n_bins_rq1"]):
    # Part 1: raw signal strength
    signal_cols = [level_col] + TEMPORAL_DYNAMICS_FEATURES
    strength_rows = []
    for col in signal_cols:
        grouped, stats_dict = analyze_signal_vs_target(train_df, col, n_bins=n_bins)
        grouped.to_csv(os.path.join(output_dir, f"temporal_{col}_binned_target.csv"))
        val_sub = val_df[[col, "target"]].dropna()
        val_spearman, _ = stats.spearmanr(val_sub[col], val_sub["target"])
        strength_rows.append({"feature": col, "n": stats_dict["n"], "spearman_r": stats_dict["spearman_r"], "spearman_p": stats_dict["spearman_p"], "pearson_r": stats_dict["pearson_r"], "pearson_p": stats_dict["pearson_p"], "spearman_r_val": val_spearman, "top_minus_bottom_bin_bps": grouped["mean"].iloc[-1] - grouped["mean"].iloc[0],})
    summary_df = pd.DataFrame(strength_rows).set_index("feature")
    summary_df.to_csv(os.path.join(output_dir, "temporal_signal_strength.csv"))

    # Part 2: incremental value over the level-only baseline
    variants = {"level_only": [level_col]}
    for feat in TEMPORAL_DYNAMICS_FEATURES:
        variants[f"level + {feat}"] = [level_col, feat]
    variants["level + all_temporal"] = [level_col] + TEMPORAL_DYNAMICS_FEATURES

    variant_rows = []
    for name, cols in variants.items():
        metrics, coef_df, _, _, _ = fit_ridge_and_evaluate(train_df, val_df, cols, name, standardize=True, alpha=CONFIG["ridge_alpha"],)
        added = [c for c in cols if c != level_col]
        added_coef = (coef_df.set_index("feature").loc[added[0], "coefficient"] if len(added) == 1 else np.nan)
        variant_rows.append({**metrics, "n_features": len(cols), "added_feature_coef": added_coef})
    variant_df = pd.DataFrame(variant_rows).set_index("model")
    base_mae = variant_df.loc["level_only", "MAE"]
    variant_df["MAE_improvement_vs_level_only"] = base_mae - variant_df["MAE"]
    variant_df["MAE_improvement_pct"] = 100 * variant_df["MAE_improvement_vs_level_only"] / base_mae
    variant_df.to_csv(os.path.join(output_dir, "temporal_incremental_model_comparison.csv"))
    plot_df = variant_df.drop(index="level_only")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.barh(plot_df.index[::-1], plot_df["MAE_improvement_vs_level_only"].values[::-1], color="#3b6fa0")
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_xlabel("MAE[level_only] - MAE[variant]  (bps; > 0 = temporal feature helps)")
    ax.set_title("Incremental value of temporal features over the imbalance level (Ridge, val)")
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, "temporal_incremental_value.png"), dpi=120)
    plt.close(fig)

    return {"signal_strength": summary_df, "model_comparison": variant_df}

In [12]:
# FEATURE IMPORTANCE DIAGNOSTICS
 
def _sample_rows(df, n, random_state):
    rng = np.random.RandomState(random_state)
    idx = np.sort(rng.choice(len(df), size=n, replace=False))
    return df.iloc[idx]
  
def _signal_family(signal, feature_cols):
    return [c for c in (signal, f"{signal}_cs_z", f"{signal}_cs_rank") if c in feature_cols]
 
def compute_liquidity_tiers(train_df, n_tiers=CONFIG["regime_n_tiers"]):
    stock_liquidity = train_df.groupby("stock_id")["matched_size"].mean().dropna()
    codes = pd.qcut(stock_liquidity, q=n_tiers, labels=False, duplicates="drop")
    return "liq_T" + (codes + 1).astype(int).astype(str)
 
def run_feature_importance_diagnostics(model, feature_cols, train_df, val_df, output_dir, shap_sample_size=CONFIG["shap_sample_size"], random_state=CONFIG["shap_random_state"], n_repeats=CONFIG["n_permutation_repeats"], gain_importance_df=None):
    print("\nFEATURE IMPORTANCE DIAGNOSTICS\n")
    val_valid = val_df.dropna(subset=["target"])

    # Permutation importance (validation MAE)
    perm_rows = _sample_rows(val_valid, CONFIG["permutation_sample_size"], random_state)
    X_perm, y_perm = perm_rows[feature_cols], perm_rows["target"].values
    baseline_mae = evaluate_predictions(y_perm, model.predict(X_perm, num_iteration=model.best_iteration_), name="baseline")["MAE"]
    perm = permutation_importance(model, X_perm, y_perm, scoring="neg_mean_absolute_error", n_repeats=n_repeats, random_state=random_state)
    perm_df = pd.DataFrame({"feature": feature_cols, "importance_mean": perm.importances_mean, "importance_std": perm.importances_std,}).sort_values("importance_mean", ascending=False).reset_index(drop=True)
    perm_df["pct_of_baseline_MAE"] = 100 * perm_df["importance_mean"] / baseline_mae
    if gain_importance_df is not None:
        gain = gain_importance_df[["feature", "gain_importance"]].copy()
        gain["gain_rank"] = gain["gain_importance"].rank(ascending=False, method="min").astype(int)
        perm_df = perm_df.merge(gain, on="feature", how="left")
        rho, rho_p = stats.spearmanr(perm_df["importance_mean"], perm_df["gain_importance"])
        print(f"Rank agreement between permutation and gain importance: Spearman rho={rho:.3f} (p={rho_p:.2e})")
    perm_df.to_csv(os.path.join(output_dir, "permutation_importance.csv"), index=False)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.barh(perm_df["feature"][::-1], perm_df["importance_mean"][::-1], xerr=perm_df["importance_std"][::-1], color="#3b6fa0")
    ax.axvline(0, color="grey", lw=0.8)
    ax.set_xlabel("Increase in validation MAE when permuted (bps)")
    ax.set_title("Permutation importance (LightGBM full feature set)")
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, "permutation_importance.png"), dpi=120)
    plt.close(fig)
 
    # SHAP on a fixed-seed validation sample
    shap_rows = _sample_rows(val_valid, shap_sample_size, random_state)
    X_shap = shap_rows[feature_cols]
    explainer = shap.TreeExplainer(model.booster_)
    shap_values = explainer.shap_values(X_shap, check_additivity=False)
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    shap_df = pd.DataFrame(shap_values, columns=feature_cols)
    meta = shap_rows[["stock_id", "seconds_in_bucket", "target"]].reset_index(drop=True)
    abs_shap = shap_df.abs()
 
    global_df = pd.DataFrame({"feature": feature_cols, "mean_abs_shap": abs_shap.mean().values})
    global_df["share_pct"] = 100 * global_df["mean_abs_shap"] / global_df["mean_abs_shap"].sum()
    global_df = global_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    global_df.to_csv(os.path.join(output_dir, "shap_mean_abs_importance.csv"), index=False) 
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.barh(global_df["feature"][::-1], global_df["mean_abs_shap"][::-1], color="#3b6fa0")
    ax.set_xlabel("Mean |SHAP value| (bps)")
    ax.set_title("SHAP summary (LightGBM full feature set, validation sample)")
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, "shap_summary.png"), dpi=120)
    plt.close(fig)
 
    # SHAP importance vs. time-to-close
    window_size = CONFIG["rq2_window_size"]
    windowed = assign_time_window(meta[["seconds_in_bucket"]], window_size=window_size)
    window_start = windowed["time_window_idx"].values * window_size
    row_total = abs_shap.sum(axis=1).values
 
    time_rows = []
    for sig in CONFIG["shap_focus_signals"]:
        if sig not in feature_cols:
            continue
        own = abs_shap[sig].values
        family = shap_df[_signal_family(sig, feature_cols)].sum(axis=1).abs().values
        for w in np.unique(window_start):
            m = window_start == w
            time_rows.append({"window_start": int(w), "signal": sig, "n_rows": int(m.sum()), "mean_abs_shap_own": own[m].mean(), "mean_abs_shap_family": family[m].mean(), "family_share_of_total_pct": 100 * family[m].sum() / row_total[m].sum(),})
    time_df = pd.DataFrame(time_rows)
    time_df.to_csv(os.path.join(output_dir, "shap_importance_vs_time.csv"), index=False)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    for sig in time_df["signal"].unique():
        sub = time_df[time_df["signal"] == sig].sort_values("window_start")
        axes[0].plot(sub["window_start"], sub["mean_abs_shap_family"], marker="o", label=sig)
        axes[1].plot(sub["window_start"], sub["family_share_of_total_pct"], marker="o", label=sig)
    axes[0].set_ylabel("Mean |SHAP| of signal family (bps)")
    axes[1].set_ylabel("Share of total |SHAP| (%)")
    for ax in axes:
        ax.axhline(0, color="grey", lw=0.8)
        ax.set_xlabel("seconds_in_bucket (window start)")
    axes[0].set_title("Absolute contribution vs. time-to-close")
    axes[1].set_title("Relative contribution vs. time-to-close")
    axes[0].legend()
    fig.suptitle("SHAP importance of imbalance signals vs. time-to-close")
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, "shap_importance_vs_time.png"), dpi=120)
    plt.close(fig)
 
    # SHAP importance vs. stock liquidity (terciles fit on train) 
    tiers = compute_liquidity_tiers(train_df)
    tier_col = meta["stock_id"].map(tiers)
    known = tier_col.notna().values
    liq_wide = abs_shap[known].groupby(tier_col[known].values).mean().T
    liq_share = 100 * liq_wide / liq_wide.sum()
    liq_df = liq_wide.join(liq_share, rsuffix="_share_pct")
    liq_df["overall"] = abs_shap[known].mean()
    liq_df = liq_df.sort_values("overall", ascending=False)
    liq_df.index.name = "feature"
    liq_df.to_csv(os.path.join(output_dir, "shap_importance_vs_liquidity.csv"))

    tier_names = list(liq_wide.columns)
    tier_ctx = pd.DataFrame({"n_rows": tier_col[known].value_counts().sort_index(), "mean_abs_target_bps": meta.loc[known, "target"].abs().groupby(tier_col[known].values).mean(),})
    top = liq_df.head(CONFIG["shap_top_n_features"])
    x = np.arange(len(top))
    width = 0.8 / len(tier_names)
    fig, ax = plt.subplots(figsize=(10, 4.8))
    for i, tier in enumerate(tier_names):
        ax.bar(x + i * width, top[tier].values, width, label=tier)
    ax.set_xticks(x + width * (len(tier_names) - 1) / 2)
    ax.set_xticklabels(top.index, rotation=30, ha="right")
    ax.set_ylabel("Mean |SHAP| (bps)")
    ax.set_title("SHAP importance by stock-liquidity tier (liq_T1 = least liquid)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(output_dir, "shap_importance_vs_liquidity.png"), dpi=120)
    plt.close(fig)
    
    return {"permutation_importance": perm_df, "shap_mean_abs": global_df, "shap_vs_time": time_df, "shap_vs_liquidity": liq_df,}

In [13]:
# REGIME ANALYSIS

REGIME_AXES = {"volatility": {"value_col": "realized_vol", "agg": "last", "fallback_col": "wap_return"}, "liquidity": {"value_col": "matched_size", "agg": "mean", "fallback_col": None}, "imbalance_magnitude": {"value_col": "abs_auction_imbalance_ratio", "agg": "mean", "fallback_col": None},}
REGIME_SOURCE_COLS = ["stock_id", "date_id", "seconds_in_bucket", "realized_vol", "wap_return", "matched_size", "auction_imbalance_ratio"]
 
def predict_with_best_iteration(model, X):
    best_iter = getattr(model, "best_iteration_", None)
    return model.predict(X, num_iteration=best_iter) if best_iter else model.predict(X)

def _regime_tier_labels(n_tiers):
    names = ["low"] + ["mid"] * (n_tiers - 2) + ["high"]
    return [f"T{i + 1}_{name}" for i, name in enumerate(names)]

def _aggregate_to_group(df, value_col, group_cols, agg, fallback_col=None, time_col="seconds_in_bucket"):
    group_cols = list(group_cols)
    cols = group_cols + [time_col, value_col] + ([fallback_col] if fallback_col else [])
    sub = df[cols].sort_values(time_col, kind="mergesort")
    grouped = sub.groupby(group_cols)
    values = grouped[value_col].agg(agg)
    n_fallback = 0
    if fallback_col:
        filled = values.fillna(grouped[fallback_col].std())
        n_fallback = int(values.isna().sum() - filled.isna().sum())
        values = filled
    return values, n_fallback
 
def assign_regime_tiers(train_df, eval_df, value_col, group_cols=("stock_id", "date_id"), n_tiers=CONFIG["regime_n_tiers"], agg="mean", fallback_col=None):
    group_cols = list(group_cols)
    train_agg, n_fb_train = _aggregate_to_group(train_df, value_col, group_cols, agg, fallback_col)
    eval_agg, n_fb_eval = _aggregate_to_group(eval_df, value_col, group_cols, agg, fallback_col)
    if fallback_col:
        print(f"  {value_col}: fallback std({fallback_col}) used for {n_fb_train} train / {n_fb_eval} eval auctions")
    _, edges = pd.qcut(train_agg.dropna(), q=n_tiers, retbins=True, duplicates="drop")
    edges = np.asarray(edges, dtype=float).copy()
    edges[0], edges[-1] = -np.inf, np.inf
    labels = _regime_tier_labels(len(edges) - 1)
 
    def broadcast(df, group_values):
        tier = pd.cut(group_values, bins=edges, labels=labels, include_lowest=True)
        table = tier.rename("regime_tier").reset_index()
        merged = df[group_cols].merge(table, on=group_cols, how="left")
        return pd.Series(pd.Categorical(merged["regime_tier"], categories=labels, ordered=True), index=df.index)
 
    return broadcast(train_df, train_agg), broadcast(eval_df, eval_agg), edges

def _tier_metrics_by_date(y, pred, tier, dates):
    frame = pd.DataFrame({"date_id": dates, "tier": tier.values, "y": y, "pred": pred})
    frame = frame[frame["tier"].notna()]
    rows = []
    for (d, t), g in frame.groupby(["date_id", "tier"], observed=True):
        yy, pp = g["y"].values, g["pred"].values
        m = evaluate_predictions(yy, pp)
        z = evaluate_predictions(yy, np.zeros(len(yy)))
        rows.append({"date_id": d, "tier": t, "corr": m["corr"], "MAE_reduction_vs_zero_pct": 100 * (z["MAE"] - m["MAE"]) / z["MAE"]})
    return pd.DataFrame(rows)

def _paired_tier_test(per_date, metric, low, high):
    pivot = per_date.pivot(index="date_id", columns="tier", values=metric)
    if low not in pivot.columns or high not in pivot.columns:
        return np.nan, np.nan, np.nan, 0
    diff = (pivot[high] - pivot[low]).dropna()
    n = len(diff)
    if n < 3:
        return np.nan, np.nan, np.nan, n
    _, p = stats.ttest_1samp(diff, 0.0)
    return diff.mean(), diff.std(ddof=1) / np.sqrt(n), p, n

def run_regime_analysis(model, feature_cols, train_df, val_df, output_dir):
    print("\nH5 - REGIME ANALYSIS")
    y = val_df["target"].values
    pred = predict_with_best_iteration(model, val_df[feature_cols])
    dates = val_df["date_id"].values
    train_light = train_df[REGIME_SOURCE_COLS].copy()
    val_light = val_df[REGIME_SOURCE_COLS].copy()
    for frame in (train_light, val_light):
        frame["abs_auction_imbalance_ratio"] = frame["auction_imbalance_ratio"].abs()
    tables, per_date_tables, edge_rows, summary_rows = {}, {}, [], []
    alpha = CONFIG["regime_significance_level"]
 
    for axis, spec in REGIME_AXES.items():
        _, val_tier, edges = assign_regime_tiers(train_light, val_light, spec["value_col"], n_tiers=CONFIG["regime_n_tiers"], agg=spec["agg"], fallback_col=spec["fallback_col"],)
        labels = list(val_tier.cat.categories)
        for i, lab in enumerate(labels):
            edge_rows.append({"axis": axis, "tier": lab, "lower_edge": edges[i], "upper_edge": edges[i + 1]})
        n_unassigned = int(val_tier.isna().sum()) 
        per_date = _tier_metrics_by_date(y, pred, val_tier, dates)
        per_date_tables[axis] = per_date
        rows = []
        for lab in labels:
            m = (val_tier == lab).values
            if m.sum() == 0:
                continue
            model_m = evaluate_predictions(y[m], pred[m], name=lab)
            zero_m = evaluate_predictions(y[m], np.zeros(m.sum()), name=lab)
            n_auctions = val_light.loc[m, ["stock_id", "date_id"]].drop_duplicates().shape[0]
            rows.append({"tier": lab, "n_rows": model_m["n"], "n_auctions": n_auctions, "MAE": model_m["MAE"], "RMSE": model_m["RMSE"], "corr": model_m["corr"], "directional_accuracy": model_m["directional_accuracy"], "MAE_zero_baseline": zero_m["MAE"], "MAE_reduction_vs_zero_pct": 100 * (zero_m["MAE"] - model_m["MAE"]) / zero_m["MAE"],})
        regime_df = pd.DataFrame(rows).set_index("tier")
        regime_df["share_of_val_auctions_pct"] = 100 * regime_df["n_auctions"] / regime_df["n_auctions"].sum()
        for metric, col in (("corr", "corr_se_by_date"), ("MAE_reduction_vs_zero_pct", "MAE_reduction_se_by_date")):
            pivot = per_date.pivot(index="date_id", columns="tier", values=metric)
            regime_df[col] = (pivot.std(ddof=1) / np.sqrt(pivot.count())).reindex(regime_df.index)
        regime_df.to_csv(os.path.join(output_dir, f"regime_analysis_{axis}.csv"))
        tables[axis] = regime_df
 
        low, high = labels[0], labels[-1]
        corr_diff, corr_se, corr_p, n_dates = _paired_tier_test(per_date, "corr", low, high)
        red_diff, red_se, red_p, _ = _paired_tier_test(per_date, "MAE_reduction_vs_zero_pct", low, high)
        summary_rows.append({
            "axis": axis, "low_tier": low, "high_tier": high, "n_dates": n_dates,
            "MAE_low": regime_df.loc[low, "MAE"], "MAE_high": regime_df.loc[high, "MAE"],
            "corr_low": regime_df.loc[low, "corr"], "corr_high": regime_df.loc[high, "corr"],
            "corr_diff": corr_diff, "corr_diff_se": corr_se, "corr_diff_p": corr_p,
            "MAE_reduction_low_pct": regime_df.loc[low, "MAE_reduction_vs_zero_pct"],
            "MAE_reduction_high_pct": regime_df.loc[high, "MAE_reduction_vs_zero_pct"],
            "MAE_reduction_diff": red_diff, "MAE_reduction_diff_se": red_se, "MAE_reduction_diff_p": red_p,})
        fig, ax = plt.subplots(figsize=(7, 4.5))
        x = np.arange(len(regime_df))
        width = 0.38
        ax.bar(x - width / 2, regime_df["MAE_zero_baseline"], width, label="zero baseline", color="#b0b7c3")
        bars = ax.bar(x + width / 2, regime_df["MAE"], width, label="model", color="#3b6fa0")
        for bar, red in zip(bars, regime_df["MAE_reduction_vs_zero_pct"]):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{red:.2f}%", ha="center", va="bottom", fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels(regime_df.index)
        ax.set_ylabel("Validation MAE (bps)")
        ax.set_title(f"MAE by {axis} tier (label = % reduction vs. zero baseline)")
        ax.set_ylim(0, regime_df[["MAE", "MAE_zero_baseline"]].max().max() * 1.2)   # headroom for the legend
        ax.legend(loc="upper right", ncol=2)
        fig.tight_layout()
        fig.savefig(os.path.join(output_dir, f"regime_mae_{axis}.png"), dpi=120)
        plt.close(fig)
    pd.DataFrame(edge_rows).to_csv(os.path.join(output_dir, "regime_tier_edges.csv"), index=False)
    summary_df = pd.DataFrame(summary_rows).set_index("axis")
    summary_df.to_csv(os.path.join(output_dir, "regime_analysis_summary.csv"))
    
    return {"regime_tables": tables, "summary": summary_df, "per_date_metrics": per_date_tables}

In [14]:
# MAIN
 
def main():
    output_dir = CONFIG["output_dir"]
    # Load Data
    df = pd.read_csv(CONFIG["data_path"])
    # Data Forensics
    dataset_forensics(df, output_dir)
    # Chronological Split
    train_df, val_df, test_df = chronological_split(df, train_frac=CONFIG["train_frac"], val_frac=CONFIG["val_frac"])
    # Walk forward baseline check
    fold_metrics = []
    for fold_id, fold_train, fold_val in generate_walk_forward_folds(df, n_folds=CONFIG["n_walk_forward_folds"]):
        preds = baseline_stock_mean(fold_train, fold_val)
        m = evaluate_predictions(fold_val["target"].values, preds, name=f"fold_{fold_id}_stock_mean")
        fold_metrics.append(m)
    fold_metrics_df = pd.DataFrame(fold_metrics)
    fold_metrics_df.to_csv(os.path.join(output_dir, "walk_forward_fold_metrics.csv"), index=False)
    # Baselines
    baseline_table = run_all_baselines(train_df, val_df)
    baseline_table.to_csv(os.path.join(output_dir, "baseline_results_on_validation.csv"))
    # Feature engineering
    train_feat = add_microstructure_features(train_df)
    val_feat = add_microstructure_features(val_df)
    feature_engineering_report(train_feat, output_dir)
    # H1
    rq1_summary = run_rq1_experiment(train_feat, output_dir, n_bins=CONFIG["n_bins_rq1"])
    # H2
    rq2_results = run_rq2_experiment(train_feat, val_feat, output_dir, window_size=CONFIG["rq2_window_size"])
    # H3
    # Model 1: Ridge across feature sets
    model1_metrics, model1_coefs = run_model1(train_feat, val_feat, output_dir)
    # Cumulative feature-group ablation, ending in the full LightGBM model
    feature_ablation, feature_importance, lgbm_model = run_cumulative_feature_ablation(train_feat, val_feat, output_dir)
    # Interaction
    interaction_results = run_interaction_ablation(train_feat, val_feat, signal_col=CONFIG["interaction_signal_col"], output_dir=output_dir)
    # Final Comparison
    best_ridge_name = model1_metrics["MAE"].idxmin()
    best_ridge_mae = model1_metrics.loc[best_ridge_name, "MAE"]
    lgbm_mae = feature_ablation["MAE"].iloc[-1]
    lgbm_gain_pct = 100 * (best_ridge_mae - lgbm_mae) / best_ridge_mae
    # H4 - Temporal Dynamics
    train_temporal = add_temporal_dynamics_features(add_cross_sectional_features(add_time_features(train_feat)))
    val_temporal = add_temporal_dynamics_features(add_cross_sectional_features(add_time_features(val_feat)))
    temporal_results = run_temporal_dynamics_experiment(train_temporal, val_temporal, output_dir)
    # Feature Importance Diagnostics
    full_feature_cols = RAW_FEATURES + IMBALANCE_FEATURES + TIME_FEATURES + CROSS_SECTIONAL_FEATURES
    importance_results = run_feature_importance_diagnostics(lgbm_model, full_feature_cols, train_temporal, val_temporal, output_dir, gain_importance_df=feature_importance,)
    # H5 - Regime Analysis
    regime_results = run_regime_analysis(lgbm_model, full_feature_cols, train_temporal, val_temporal, output_dir)
    
    return {
        "train_df": train_df, "val_df": val_df, "test_df": test_df,
        "train_feat": train_feat, "val_feat": val_feat,
        "baseline_table": baseline_table, "rq1_summary": rq1_summary,
        "fold_metrics": fold_metrics_df, "rq2_results": rq2_results,
        "model1_metrics": model1_metrics, "model1_coefs": model1_coefs,
        "feature_ablation": feature_ablation, "feature_importance": feature_importance,
        "interaction_results": interaction_results, "lgbm_model": lgbm_model,
        "train_temporal": train_temporal, "val_temporal": val_temporal,
        "temporal_results": temporal_results,
        "full_feature_cols": full_feature_cols,
        "importance_results": importance_results,
        "regime_results": regime_results,}

if __name__ == "__main__":
    results = main()

Shape: (5237980, 17)

N stocks      : 200
N dates       : 481
N observations: 5237980
Unique keys: 5237980
Rows: 5237980

near_price missing rate by seconds_in_bucket (first/last 5 buckets):
seconds_in_bucket
0     1.0
10    1.0
20    1.0
30    1.0
40    1.0
seconds_in_bucket
500    0.000042
510    0.000042
520    0.000042
530    0.000042
540    0.000042

Extreme observations: 56236 (1.074% of rows), bounds=(-31.47, 31.32)

Chronological split:
train dates: [0, 335]
val dates: [336, 407]
test dates: [408, 480]

Fold 1: train dates [0,239] (240 dates) -> val dates [240,287] (48 dates)
Fold 2: train dates [0,287] (288 dates) -> val dates [288,335] (48 dates)
Fold 3: train dates [0,335] (336 dates) -> val dates [336,383] (48 dates)
Fold 4: train dates [0,383] (384 dates) -> val dates [384,431] (48 dates)

CUMULATIVE FEATURE-GROUP ABLATION

raw                n_features= 11  MAE=6.40063
+ imbalance        n_features= 17  MAE=6.38818  delta_MAE=-0.01245
+ time             n_features= 19  MA